# Analysis and comparison of clusterprep training runs

This notebook reads saved results without retraining models. It lets you list and select runs, explore each run, rank models, and measure differences **with / without Chandra for the same model family**.

**Usage:** run the cells in order; select runs in the list (Ctrl/Cmd or Shift for multiple selection), then rerun the cells starting from “Apply selection”. An empty `SELECTED_RUNS` list selects all completed runs. To inspect an incomplete run, select it explicitly and use the detailed analysis section.

Dependencies: `numpy pandas matplotlib scikit-learn ipython`; `ipywidgets` is optional. Use the project Python kernel (`.venv`). The adjacent `training_analysis.py` file contains the analysis functions.

In [ ]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

# Locate the project from its root or clusterprep/notebooks.
PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                     if (p / 'clusterprep/notebooks/training_analysis.py').exists()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Open the notebook from PH_Project_I or set PROJECT_ROOT.")
sys.path.insert(0, str(PROJECT_ROOT / 'clusterprep/notebooks'))
from training_analysis import (discover, comparison, rank_table, plot_comparison,
    detail, chandra_pairs, paired_delta, protocol_summary, gain_tables, plot_gain_heatmap, METRICS, LOWER)
plt.rcParams.update({'figure.dpi': 100, 'axes.spines.top': False, 'axes.spines.right': False})
pd.set_option('display.max_colwidth', 100)

RESULT_ROOTS = [PROJECT_ROOT / 'results/clusterprep']
# Add PROJECT_ROOT / 'results/experiments' to include older results.
SELECTED_RUNS = []  # Exact directory names or absolute paths.
MODEL_FILTER = []   # Example: ['cnn', 'image_cnn']
MODE_FILTER = []    # Example: ['raw', 'multimodal']
RANK_METRIC = 'macro_f1'  # Also accuracy, balanced_accuracy, roc_auc_macro, log_loss...
COHORT = 'all'      # all, paired (= Chandra observation available), raw_only
COMMON_SAMPLES = False  # True: recompute on the intersection of selected runs.
EXPORT = False
EXPORT_DIR = PROJECT_ROOT / 'results/analysis_clusterprep'


## 1. Run inventory and selection

Failed or incomplete runs remain visible but are excluded from the ranking. Invalid files are reported. The availability of a Chandra observation (`has_chandra`) differs from its use: a `raw` model can be evaluated on a cluster that also has a Chandra observation.

In [ ]:
runs, inventory, loading_issues = discover(RESULT_ROOTS)
if inventory.empty:
    raise RuntimeError("No clusterprep training runs found: check RESULT_ROOTS.")
display(inventory)
if not loading_issues.empty:
    display(loading_issues)
print(f"{len(inventory)} training runs found")

eligible = inventory.copy()
if MODEL_FILTER:
    eligible = eligible[eligible.model.isin(MODEL_FILTER)]
if MODE_FILTER:
    eligible = eligible[eligible['mode'].isin(MODE_FILTER)]
if SELECTED_RUNS:
    initial = eligible[eligible.run_id.isin(SELECTED_RUNS) | eligible.name.isin(SELECTED_RUNS)].run_id.tolist()
    unmatched = set(SELECTED_RUNS) - set(eligible.run_id) - set(eligible.name)
    if unmatched:
        raise ValueError(f'Unknown or filtered runs: {unmatched}')
else:
    initial = eligible.loc[eligible.status == 'completed', 'run_id'].tolist()
selector = None
try:
    import ipywidgets as widgets
    selector = widgets.SelectMultiple(
        options=[(f"{r.model} | {r['mode']} | {r['name']} [{r.status}]", r.run_id)
                 for _, r in eligible.iterrows()],
        value=tuple(initial), rows=min(16, max(3, len(eligible))),
        description='Runs', layout=widgets.Layout(width='100%'))
    display(selector)
except ImportError:
    print('ipywidgets unavailable: edit SELECTED_RUNS in the configuration.')


### Apply selection

After changing the list, rerun this cell and the following cells. Configuration differences are displayed to help interpret the results.

In [ ]:
selected_ids = list(selector.value) if selector is not None else initial
if not selected_ids:
    raise ValueError('Select at least one training run.')
selected = [runs[k] for k in selected_ids]
print(f'{len(selected)} training runs selected')
config_table = pd.DataFrame({r.path.name: r.config for r in selected}).T
variable_columns = [c for c in config_table if config_table[c].map(lambda v: json.dumps(v, sort_keys=True)).nunique() > 1]
display(config_table[variable_columns] if variable_columns else config_table)


## 2. Ranking and model selection

Ranking uses **validation** results and `RANK_METRIC` (macro F1 by default, suitable for imbalanced classes). Metrics are recomputed from probabilities using the saved class order. Log loss is unweighted and has no label smoothing; it may differ from the training `loss`. The Brier score sums squared errors across classes.

A separate ranking is produced for each cohort: identical keys, labels, clusters, class order, and source file signatures. Runs with unverifiable provenance remain separate. `COMMON_SAMPLES=True` restricts evaluation to shared samples; this changes the target population and must be stated in the report. A cohort containing only one class has undefined macro AUC/AP.

Rank 1 identifies a **candidate according to the chosen criterion**, not statistical evidence of superiority. Ties are retained. Test results then describe generalization; selecting a model after repeatedly inspecting the test set requires a new independent evaluation.

In [ ]:
validation_table, validation_issues, validation_frames = comparison(
    selected, split='validation', cohort=COHORT, common=COMMON_SAMPLES)
ranking = rank_table(validation_table, RANK_METRIC)
display(ranking)
if not validation_issues.empty:
    display(validation_issues)
plot_comparison(ranking, RANK_METRIC)
if not ranking.empty:
    candidates = ranking[ranking.rank_in_cohort == 1]
    print('Top candidates, separately for each cohort:')
    display(candidates[['name', 'cohort_id', 'n', RANK_METRIC]])
    if ranking.cohort_id.nunique() > 1:
        print('Multiple cohorts: no overall best model is declared.')


In [ ]:
test_table, test_issues, test_frames = comparison(
    selected, split='test', cohort=COHORT, common=COMMON_SAMPLES)
display(test_table)
if not test_issues.empty:
    display(test_issues)
plot_comparison(test_table, RANK_METRIC)
# Cost and size: consider practical constraints when selecting a model.
display(inventory[inventory.run_id.isin(selected_ids)][
    ['name', 'model', 'mode', 'parameters', 'minutes', 'epochs', 'best_epoch']])


### Summary across splits / repeated runs

Configurations are grouped by identical hyperparameters, mode, code, and normalization (excluding seed). Repetitions within each cohort are averaged first, followed by averaging across cohorts: repeating a cohort more often does not give it more weight. Dispersion is the standard deviation across cohorts, **not** a confidence interval. Cohorts may overlap and are not assumed independent.

The aggregate ranking only compares configurations covering **exactly the same set of cohorts**. Missing metrics are not replaced with zero; a configuration with a missing metric in any cohort is not ranked for that criterion. For specific scientific objectives, also examine per-class performance in the detailed analysis.

In [ ]:
validation_protocols = protocol_summary(validation_table, selected, RANK_METRIC)
test_protocols = protocol_summary(test_table, selected, RANK_METRIC)
summary_columns = ['model', 'mode', 'n_runs', 'n_cohorts', 'seeds', 'cohort_set',
                   f'{RANK_METRIC}_mean', f'{RANK_METRIC}_std', 'rank_in_cohort_set']
print('Validation: comparison of configurations on the same cohorts')
display(validation_protocols[summary_columns] if not validation_protocols.empty else validation_protocols)
print('Test: aggregate performance summary')
display(test_protocols.drop(columns=['rank_in_cohort_set'], errors='ignore'))

## 3. Detailed analysis of each training run

Select a run and click **Analyze**. Without widgets, edit `DETAIL_RUN`. `ANALYZE_ALL=True` generates reports for all selected runs (large output). Diagnostics use the `DETAIL_SPLIT` split.

Class counts and Chandra availability help identify population bias. A difference between the `paired` and `raw_only` cohorts within one run **does not measure the contribution of Chandra**, because the clusters differ. Learning curves show training metrics measured with augmentation/mixup enabled, which can affect the train/validation gap.

In [ ]:
DETAIL_RUN = selected_ids[0]  # Or the exact directory name.
DETAIL_SPLIT = 'test'
ANALYZE_ALL = False

def show_detail(run):
    display(Markdown(f'### {run.path.name}'))
    try:
        detail(run, split=DETAIL_SPLIT, display_fn=display)
    except (ValueError, OSError) as exc:
        print(f'Partial analysis: {exc}')

if ANALYZE_ALL:
    for run in selected:
        show_detail(run)
elif selector is not None:
    detail_choice = widgets.Dropdown(options=[(r.path.name, r.id) for r in selected],
                                    value=DETAIL_RUN, layout=widgets.Layout(width='100%'))
    detail_button = widgets.Button(description='Analyze', button_style='primary')
    detail_output = widgets.Output()
    rendered_detail = [None]
    def update_detail(_):
        detail_key = (detail_choice.value, DETAIL_SPLIT)
        if detail_key == rendered_detail[0]:
            return
        rendered_detail[0] = detail_key
        with detail_output:
            detail_output.clear_output(wait=False)
            show_detail(runs[detail_choice.value])
    detail_button.on_click(update_detail)
    display(detail_choice, detail_button, detail_output)
    update_detail(None)
else:
    show_detail(next(r for r in selected if DETAIL_RUN in [r.id, r.path.name]))


## 4. Chandra contribution for the same model: protocol checks

Candidate pairs match each selected `pair` or `multimodal` run to selected `raw` runs from the **same model family**. All candidates are shown; the raw baseline is not chosen retrospectively to maximize the gain.

A pair is controlled if hyperparameters (including seed), train/validation/test splits, sources, normalization, software versions, and saved code match. Mode and operational settings may differ. The channel/parameter count may change with Chandra: this comparison measures the benefit of the Chandra pipeline, not an ablation with fixed weights. Metadata does not guarantee GPU determinism.

**`pair` uses Chandra**; it is not a radio-only baseline. If a model has no `raw` run, the contribution cannot be estimated from these artifacts; a compatible baseline must be trained, or a separate input ablation must be performed on the checkpoint. This notebook does not generate that baseline.

In [ ]:
pair_catalog = chandra_pairs(selected)
display(pair_catalog)
controlled_pairs = (pair_catalog[pair_catalog.controlled & pair_catalog.raw_id.notna()]
                    if not pair_catalog.empty else pd.DataFrame())
print(f'{len(controlled_pairs)} controlled pairs available')
# Descriptive deltas for all controlled pairs and the three cohorts.
delta_tables, delta_issues = [], []
for _, pair in controlled_pairs.iterrows():
    for cohort in ['all', 'paired', 'raw_only']:
        try:
            result = paired_delta(runs[pair.raw_id], runs[pair.chandra_id], cohort=cohort, n_boot=0)
            result['raw_id'], result['chandra_id'] = pair.raw_id, pair.chandra_id
            delta_tables.append(result)
        except (ValueError, OSError) as exc:
            delta_issues.append({'raw_id': pair.raw_id, 'chandra_id': pair.chandra_id,
                                 'cohort': cohort, 'error': str(exc)})
chandra_deltas = pd.concat(delta_tables, ignore_index=True) if delta_tables else pd.DataFrame()
display(chandra_deltas)
if delta_issues:
    display(pd.DataFrame(delta_issues))


### Chandra gain heatmap

One row per `multimodal/raw` or `pair/raw` comparison for the same model, and one column per metric. **Blue = improvement; red = decrease; gray = undefined metric.** Each cell is the Chandra score minus the radio baseline score, recomputed on the same clusters. A delta of `+0.10` means 10 percentage points.

`GAIN_COHORT='paired'` compares clusters with Chandra observations; choose `all` for the full intersection of the two evaluation sets. For `pair/raw`, this intersection remains limited to clusters in the `pair` run. `GAIN_SPLIT` selects test or validation.

The `f1_de`, `recall_de`, `precision_de`, `roc_auc`, and `pr_auc` columns use **DE as the positive class**, regardless of its index. Here, `pr_auc` means **average precision (AP)**, not trapezoidal integration of the PR curve. DE metrics are undefined when DE is absent; ROC/AP require both classes.

Pairs are identified from splits and protocol, without inferring a fold from the date. The `p` number refers to the catalog and mapping table below, which contains exact paths and absolute scores. Multiple compatible baselines produce multiple rows.

For `pair/raw`, filtering the shared base splits by Chandra availability is allowed; all other checks remain active. An asterisk `*` then indicates a different training population: the delta does not isolate the effect of Chandra. Models with no raw baseline and incompatible pairs appear in the exclusions. No comparisons between different architectures are created.

`GAIN_ALLOW_CODE_CHANGE_MODELS = ('dual_encoder_cnn',)` explicitly includes this model when both runs have recorded but different code signatures. All other checks remain active (model family, class order, hyperparameters, source files, software versions, normalization, and compatible splits). Set this tuple to `()` to restore strict code matching.

Rows labeled `[code change]` are descriptive comparisons across code versions, not fully controlled ablations. Their `controlled` flag remains false. The mapping table records both code signatures and parameter counts, and retains all differences. An additional `*` indicates a changed training population for `pair/raw`. The strict pair catalog above still reports these differences; the exception applies only to this heatmap. Missing code signatures are never accepted by this option.

In [ ]:
GAIN_SPLIT = 'test'  # 'validation' or 'test'
GAIN_COHORT = 'all'  # 'all', 'paired' or 'raw_only'
INCLUDE_PAIR_RAW = True
GAIN_ALLOW_CODE_CHANGE_MODELS = ('dual_encoder_cnn',)  # Explicit descriptive comparison across code versions.
GAIN_ANNOTATE = True
GAIN_COLOR_LIMIT = None  # None: automatic symmetric scale; e.g. 0.7 to fix the range at ±0.7.

gain_matrix, gain_details, gain_issues = gain_tables(
    selected, split=GAIN_SPLIT, cohort=GAIN_COHORT, include_pair=INCLUDE_PAIR_RAW,
    allow_code_change_models=GAIN_ALLOW_CODE_CHANGE_MODELS)
gain_figure = None
if gain_matrix.empty:
    print('No compatible comparisons for this selection/cohort. See exclusions.')
else:
    gain_figure = plot_gain_heatmap(
        gain_matrix,
        title=f'X-ray gain per run (blue = positive) — {GAIN_SPLIT}, {GAIN_COHORT}',
        annotate=GAIN_ANNOTATE, limit=GAIN_COLOR_LIMIT)
    plt.show()
    display(gain_matrix.style.format('{:+.3f}', na_rep='N/A'))
    print('Row mapping, protocols, and absolute scores:')
    display(gain_details)
    if gain_details.code_changed.any():
        print('[code change]: descriptive comparison across different recorded code versions.')
if not gain_issues.empty:
    print('Missing baselines or excluded comparisons:')
    display(gain_issues)


### Uncertainty interval for a selected pair

Choose `PAIR_INDEX` from the catalog (index shown on the left). Default: the first controlled pair. The bootstrap resamples **clusters** in a paired manner and recomputes metrics using exactly the same draws for both models. The 95% percentile interval describes sampling uncertainty conditional on the two trained models; it does not cover variability across seeds or multiple comparisons.

`delta = with Chandra − without Chandra`. Positive deltas are favorable for scores; negative deltas are favorable for log loss/Brier. Score deltas × 100 give **percentage points**, not relative improvement. Draws with undefined AUC are excluded for that metric, and the valid count is reported. An interval containing zero is inconclusive. Interpret intervals cautiously when few clusters are available.

Uncontrolled pairs are rejected by default. `ALLOW_EXPLORATORY=True` allows a descriptive comparison on shared clusters with differences displayed; the gain cannot then be attributed to Chandra alone.

In [ ]:
PAIR_INDEX = None  # Example: 0, 1, ... (pair_catalog index)
PAIR_COHORT = 'paired'  # Compare the same clusters with Chandra observations.
BOOTSTRAP = 500        # 2000+ for a final report.
ALLOW_EXPLORATORY = False
paired_uncertainty = pd.DataFrame()
if PAIR_INDEX is None and not controlled_pairs.empty:
    PAIR_INDEX = controlled_pairs.index[0]
if PAIR_INDEX is None:
    print('No controlled pair: check the catalog and select a compatible raw baseline.')
else:
    pair = pair_catalog.loc[PAIR_INDEX]
    display(pair.to_frame('selected pair'))
    if pd.isna(pair.raw_id):
        print('No baseline without Chandra available.')
    elif not pair.controlled and not ALLOW_EXPLORATORY:
        print('Comparison rejected: different protocol. See differences.')
    else:
        if not pair.controlled:
            print('EXPLORATORY: Chandra effect not isolated —', pair.differences)
        try:
            paired_uncertainty = paired_delta(runs[pair.raw_id], runs[pair.chandra_id],
                                              cohort=PAIR_COHORT, n_boot=BOOTSTRAP)
            display(paired_uncertainty)
            fig, ax = plt.subplots(figsize=(10, 5))
            for i, row in paired_uncertainty.iterrows():
                ax.plot([row.ci_low, row.ci_high], [i, i], color='steelblue', lw=3)
                ax.plot(row.delta, i, 'o', color='black')
            ax.axvline(0, color='gray', ls='--')
            ax.set(yticks=range(len(paired_uncertainty)), yticklabels=paired_uncertainty.metric,
                   xlabel='Delta with − without Chandra; 95% percentile interval',
                   title=f'Paired comparison — {PAIR_COHORT}')
            fig.tight_layout()
            plt.show()
        except (ValueError, OSError) as exc:
            print(f'Comparison unavailable: {exc}')


## 5. Reproducible exports and interpretation

Set `EXPORT=True` in the configuration and rerun this cell. Tables are exported to a directory separate from the training runs. The exact selection, criterion, cohort, and bootstrap settings are recorded. The gain heatmap is exported as PNG and PDF, alongside its CSV table.

To select a model, first examine validation candidates on the same cohort, then per-class recall/F1, calibration, learning-curve stability, cost, and finally test results. Repeat training with multiple seeds before claiming robust superiority. Repetitions with the same seed are not independent folds.

In [ ]:
if EXPORT:
    EXPORT_DIR.mkdir(parents=True, exist_ok=True)
    tables = {'xray_gain_details': gain_details, 'xray_gain_issues': gain_issues,
              'xray_gain': gain_matrix.rename_axis('comparison').reset_index(),
              'inventory': inventory, 'ranking_validation': ranking, 'test': test_table,
              'configurations': config_table.reset_index(names='run_name'), 'chandra_pairs': pair_catalog,
              'chandra_deltas': chandra_deltas, 'paired_uncertainty': paired_uncertainty,
              'loading_issues': loading_issues, 'validation_issues': validation_issues,
              'validation_protocols': validation_protocols, 'test_protocols': test_protocols,
              'test_issues': test_issues, 'chandra_issues': pd.DataFrame(delta_issues)}
    for name, table in tables.items():
        table.to_csv(EXPORT_DIR / f'{name}.csv', index=False)
    if gain_figure is not None:
        gain_figure.savefig(EXPORT_DIR / 'xray_gain.png', dpi=180, bbox_inches='tight')
        gain_figure.savefig(EXPORT_DIR / 'xray_gain.pdf', bbox_inches='tight')
    manifest = dict(gain_allow_code_change_models=list(GAIN_ALLOW_CODE_CHANGE_MODELS),
                    gain_split=GAIN_SPLIT, gain_cohort=GAIN_COHORT,
                    include_pair_raw=INCLUDE_PAIR_RAW, gain_color_limit=GAIN_COLOR_LIMIT,
                    selected_runs=selected_ids, roots=list(map(str, RESULT_ROOTS)),
                    rank_metric=RANK_METRIC, cohort=COHORT, common_samples=COMMON_SAMPLES,
                    pair_index=None if PAIR_INDEX is None else int(PAIR_INDEX), pair_cohort=PAIR_COHORT,
                    bootstrap=BOOTSTRAP, bootstrap_seed=42, allow_exploratory=ALLOW_EXPLORATORY)
    (EXPORT_DIR / 'selection.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
    print(f'Reports exported: {EXPORT_DIR}')
else:
    print('Export disabled. Set EXPORT=True to save the tables.')
